In [1]:
# ============================================================
# CELL 1: IMPORT + CẤU HÌNH ĐƯỜNG DẪN
# ============================================================

import os
import gc
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")

# Output của notebook tiền xử lý
PREP_INPUT_DIR = "/kaggle/input/notebooks/luminhanh/ba-model-prep-data"
WORKING_DIR = "/kaggle/working"

SEED = 2026

print("LightGBM training notebook - dùng trực tiếp output tiền xử lý")
print("PREP_INPUT_DIR:", PREP_INPUT_DIR)
print("WORKING_DIR:", WORKING_DIR)

# Tìm file trong input dir, kể cả nếu Kaggle lồng thêm folder con
def find_file(filename, root=PREP_INPUT_DIR):
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    raise FileNotFoundError(f"Không tìm thấy {filename} trong {root}")

required_files = [
    "X_train.parquet",
    "y_train.parquet",
    "X_val.parquet",
    "y_val.parquet",
    "X_test.parquet",
    "feature_cols.pkl",
    "target_cols.pkl",
    "test_id_matrix.pkl",
    "test_dates.pkl",
]

print("\n📦 Kiểm tra file input từ notebook tiền xử lý:")
file_paths = {}
for fname in required_files:
    try:
        file_paths[fname] = find_file(fname)
        print("✅", fname, "->", file_paths[fname])
    except FileNotFoundError as e:
        print("❌", e)

gc.collect()

LightGBM training notebook - dùng trực tiếp output tiền xử lý
PREP_INPUT_DIR: /kaggle/input/notebooks/luminhanh/ba-model-prep-data
WORKING_DIR: /kaggle/working

📦 Kiểm tra file input từ notebook tiền xử lý:
✅ X_train.parquet -> /kaggle/input/notebooks/luminhanh/ba-model-prep-data/X_train.parquet
✅ y_train.parquet -> /kaggle/input/notebooks/luminhanh/ba-model-prep-data/y_train.parquet
✅ X_val.parquet -> /kaggle/input/notebooks/luminhanh/ba-model-prep-data/X_val.parquet
✅ y_val.parquet -> /kaggle/input/notebooks/luminhanh/ba-model-prep-data/y_val.parquet
✅ X_test.parquet -> /kaggle/input/notebooks/luminhanh/ba-model-prep-data/X_test.parquet
✅ feature_cols.pkl -> /kaggle/input/notebooks/luminhanh/ba-model-prep-data/feature_cols.pkl
✅ target_cols.pkl -> /kaggle/input/notebooks/luminhanh/ba-model-prep-data/target_cols.pkl
✅ test_id_matrix.pkl -> /kaggle/input/notebooks/luminhanh/ba-model-prep-data/test_id_matrix.pkl
✅ test_dates.pkl -> /kaggle/input/notebooks/luminhanh/ba-model-prep-data/te

90

In [2]:
# ============================================================
# CELL 2: LOAD X/Y/TEST_ID TỪ NOTEBOOK TIỀN XỬ LÝ
# ============================================================

print("🚀 Đang load dữ liệu đã tiền xử lý...")

X_train = pd.read_parquet(file_paths["X_train.parquet"])
y_train = pd.read_parquet(file_paths["y_train.parquet"]).values.astype(np.float32)

X_val = pd.read_parquet(file_paths["X_val.parquet"])
y_val = pd.read_parquet(file_paths["y_val.parquet"]).values.astype(np.float32)

X_test = pd.read_parquet(file_paths["X_test.parquet"])

feature_cols = pd.read_pickle(file_paths["feature_cols.pkl"]).tolist()
target_cols = pd.read_pickle(file_paths["target_cols.pkl"]).tolist()

test_id_matrix = pd.read_pickle(file_paths["test_id_matrix.pkl"])
test_dates = pd.read_pickle(file_paths["test_dates.pkl"])

if hasattr(test_dates, "tolist"):
    test_dates = test_dates.tolist()

print("✅ Loaded!")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("test_id_matrix:", test_id_matrix.shape)
print("Số feature:", len(feature_cols))
print("Target cols:", target_cols)

# Align cột cho chắc chắn
X_train = X_train.reindex(columns=feature_cols)
X_val = X_val.reindex(columns=feature_cols)
X_test = X_test.reindex(columns=feature_cols)

if y_train.ndim != 2 or y_train.shape[1] != 16:
    raise ValueError(f"y_train phải có shape (n_rows, 16), hiện tại: {y_train.shape}")

if y_val.ndim != 2 or y_val.shape[1] != 16:
    raise ValueError(f"y_val phải có shape (n_rows, 16), hiện tại: {y_val.shape}")

if len(X_train) != len(y_train):
    raise ValueError(f"X_train/y_train lệch dòng: {len(X_train)} vs {len(y_train)}")

if len(X_val) != len(y_val):
    raise ValueError(f"X_val/y_val lệch dòng: {len(X_val)} vs {len(y_val)}")

gc.collect()

🚀 Đang load dữ liệu đã tiền xử lý...
✅ Loaded!
X_train: (1005090, 633)
y_train: (1005090, 16)
X_val: (167515, 633)
y_val: (167515, 16)
X_test: (167515, 633)
test_id_matrix: (167515, 16)
Số feature: 633
Target cols: ['target_day_1', 'target_day_2', 'target_day_3', 'target_day_4', 'target_day_5', 'target_day_6', 'target_day_7', 'target_day_8', 'target_day_9', 'target_day_10', 'target_day_11', 'target_day_12', 'target_day_13', 'target_day_14', 'target_day_15', 'target_day_16']


0

In [3]:
# ============================================================
# CELL 3: CHUẨN HOÁ DTYPE + WEIGHT + METRIC
# ============================================================

print("🚀 Chuẩn hoá dữ liệu cho LightGBM...")

def prepare_lgb_input(df):
    df = df.copy()

    for col in df.columns:
        if str(df[col].dtype) == "category":
            df[col] = df[col].cat.codes.astype("int32")
        elif df[col].dtype == "object":
            df[col] = pd.factorize(df[col], sort=True)[0].astype("int32")

    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].astype("float32")
        elif pd.api.types.is_integer_dtype(df[col]):
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= 0:
                if c_max <= np.iinfo(np.uint8).max:
                    df[col] = df[col].astype("uint8")
                elif c_max <= np.iinfo(np.uint16).max:
                    df[col] = df[col].astype("uint16")
                elif c_max <= np.iinfo(np.uint32).max:
                    df[col] = df[col].astype("uint32")
                else:
                    df[col] = df[col].astype("int32")
            else:
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype("int8")
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype("int16")
                else:
                    df[col] = df[col].astype("int32")

    return df


def make_weight(df):
    if "perishable" in df.columns:
        return (df["perishable"].astype(np.float32).values * 0.25 + 1.0).astype(np.float32)
    return np.ones(len(df), dtype=np.float32)


def nwrmsle_log(y_true_log, y_pred_log, weight):
    y_true_log = np.asarray(y_true_log, dtype=np.float32)
    y_pred_log = np.asarray(y_pred_log, dtype=np.float32)
    weight = np.asarray(weight, dtype=np.float32)

    err = (y_true_log - y_pred_log) ** 2
    err = err.sum(axis=1) * weight
    return float(np.sqrt(err.sum() / weight.sum() / y_true_log.shape[1]))


X_train_model = prepare_lgb_input(X_train)
X_val_model = prepare_lgb_input(X_val)
X_test_model = prepare_lgb_input(X_test)

X_val_model = X_val_model.reindex(columns=X_train_model.columns, fill_value=0)
X_test_model = X_test_model.reindex(columns=X_train_model.columns, fill_value=0)

train_weight = make_weight(X_train_model)
val_weight = make_weight(X_val_model)

CATE_VARS = []

print("✅ Sau chuẩn hoá:")
print("X_train_model:", X_train_model.shape)
print("X_val_model:", X_val_model.shape)
print("X_test_model:", X_test_model.shape)
print("train_weight:", train_weight.shape)
print("val_weight:", val_weight.shape)

gc.collect()

🚀 Chuẩn hoá dữ liệu cho LightGBM...
✅ Sau chuẩn hoá:
X_train_model: (1005090, 633)
X_val_model: (167515, 633)
X_test_model: (167515, 633)
train_weight: (1005090,)
val_weight: (167515,)


0

In [4]:
# ============================================================
# CELL 4: BASE PARAMS LIGHTGBM 
# ============================================================

BASE_PARAMS = {
    "objective": "regression",
    "metric": "l2",

    # Tham số LightGBM cơ bản
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 100,

    # Sampling cơ bản
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,

    # Regularization cơ bản
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "min_gain_to_split": 0.0,

    # Khác
    "max_bin": 255,
    "verbosity": -1,
    "num_threads": -1,
    "force_col_wise": True,

    # Reproducible
    "seed": SEED,
    "feature_fraction_seed": SEED,
    "bagging_seed": SEED,
    "data_random_seed": SEED,
}

# Cấu hình train chính
MAX_ROUNDS = 4000
EARLY_STOPPING_ROUNDS = 150

print("✅ BASE_PARAMS - tham số cơ bản:")
for k, v in BASE_PARAMS.items():
    print(f"{k}: {v}")

print("\nMAX_ROUNDS:", MAX_ROUNDS)
print("EARLY_STOPPING_ROUNDS:", EARLY_STOPPING_ROUNDS)

✅ BASE_PARAMS - tham số cơ bản:
objective: regression
metric: l2
learning_rate: 0.05
num_leaves: 31
max_depth: -1
min_data_in_leaf: 100
feature_fraction: 0.9
bagging_fraction: 0.9
bagging_freq: 1
lambda_l1: 0.0
lambda_l2: 0.0
min_gain_to_split: 0.0
max_bin: 255
verbosity: -1
num_threads: -1
force_col_wise: True
seed: 2026
feature_fraction_seed: 2026
bagging_seed: 2026
data_random_seed: 2026

MAX_ROUNDS: 4000
EARLY_STOPPING_ROUNDS: 150


In [5]:
# ============================================================
# CELL 5: LOAD OPTUNA 
# ============================================================

import os
import json
import gc
import numpy as np

# File Optuna đã tune sẵn từ dataset input
OPTUNA_PARAM_PATH = "/kaggle/input/datasets/hailehong/optuna/optuna_best_params.json"

# Nếu không có file trên thì mới chạy Optuna
RUN_OPTUNA_IF_NO_FILE = True

# Cấu hình Optuna fallback
N_TRIALS = 20
OPTUNA_MAX_ROUNDS = 1800
OPTUNA_EARLY_STOPPING = 100

# Tune đại diện 3 vùng horizon:
# 0  = ngắn hạn, day +1
# 6  = trung hạn, day +7
# 13 = dài hạn, day +14
TUNE_HORIZONS = [0, 6, 13]

OPTUNA_SAMPLE_ROWS = 300_000


# ============================================================
# CASE 1: ĐÃ CÓ FILE OPTUNA PARAMS → LOAD VÀ DÙNG LUÔN
# ============================================================

if os.path.exists(OPTUNA_PARAM_PATH):
    print("✅ Tìm thấy file Optuna params đã tune sẵn.")
    print("📌 Đường dẫn:", OPTUNA_PARAM_PATH)

    with open(OPTUNA_PARAM_PATH, "r") as f:
        OPTUNA_BEST_PARAMS = json.load(f)

    # Đảm bảo các tham số bắt buộc đúng với notebook hiện tại
    OPTUNA_BEST_PARAMS.update({
        "objective": "regression",
        "metric": "l2",
        "verbosity": -1,
        "num_threads": -1,
        "force_col_wise": True,

        "seed": SEED,
        "feature_fraction_seed": SEED,
        "bagging_seed": SEED,
        "data_random_seed": SEED,
    })

    print("\n✅ OPTUNA_BEST_PARAMS được load từ file:")
    for k, v in OPTUNA_BEST_PARAMS.items():
        print(f"{k}: {v}")

    # Lưu copy vào working để output/commit nếu cần
    out_path = os.path.join(WORKING_DIR, "optuna_best_params_loaded.json")
    with open(out_path, "w") as f:
        json.dump(OPTUNA_BEST_PARAMS, f, indent=2)

    print("\n💾 Đã lưu bản copy tại:")
    print(out_path)

    gc.collect()


# ============================================================
# CASE 2: CHƯA CÓ FILE → CHẠY OPTUNA TỪ BASE_PARAMS
# ============================================================

else:
    print("⚠️ Không tìm thấy file Optuna params tại:")
    print(OPTUNA_PARAM_PATH)

    if not RUN_OPTUNA_IF_NO_FILE:
        raise FileNotFoundError(
            "Không có optuna_best_params.json và RUN_OPTUNA_IF_NO_FILE=False."
        )

    print("\n🚀 Bắt đầu Optuna tuning từ BASE_PARAMS...")
    print("N_TRIALS:", N_TRIALS)
    print("TUNE_HORIZONS:", TUNE_HORIZONS)
    print("OPTUNA_SAMPLE_ROWS:", OPTUNA_SAMPLE_ROWS)

    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
    except Exception as e:
        raise ImportError(
            "Không import được optuna. Nếu Kaggle chưa có optuna, "
            "hãy add file optuna_best_params.json làm input hoặc cài optuna."
        ) from e

    rng = np.random.default_rng(SEED)

    if OPTUNA_SAMPLE_ROWS is not None and OPTUNA_SAMPLE_ROWS < len(X_train_model):
        sample_idx = rng.choice(
            len(X_train_model),
            size=OPTUNA_SAMPLE_ROWS,
            replace=False
        )

        X_train_opt = X_train_model.iloc[sample_idx].reset_index(drop=True)
        y_train_opt = y_train[sample_idx]
        train_weight_opt = train_weight[sample_idx]
    else:
        X_train_opt = X_train_model
        y_train_opt = y_train
        train_weight_opt = train_weight

    print("X_train_opt:", X_train_opt.shape)
    print("y_train_opt:", y_train_opt.shape)

    def objective(trial):
        params = BASE_PARAMS.copy()

        params.update({
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.015, 0.08, log=True
            ),

            "num_leaves": trial.suggest_int("num_leaves", 24, 128),
            "max_depth": trial.suggest_int("max_depth", 5, 14),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 80, 500),

            "feature_fraction": trial.suggest_float("feature_fraction", 0.60, 1.00),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.60, 1.00),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 5),

            "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 2.0),
            "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 1.0),

            "max_bin": trial.suggest_categorical("max_bin", [127, 255]),
        })

        val_pred_tmp = np.zeros(
            (X_val_model.shape[0], len(TUNE_HORIZONS)),
            dtype=np.float32
        )

        best_iters_tmp = []

        for j, h in enumerate(TUNE_HORIZONS):
            dtrain = lgb.Dataset(
                X_train_opt,
                label=y_train_opt[:, h],
                weight=train_weight_opt,
                categorical_feature=CATE_VARS,
                free_raw_data=True
            )

            dval = lgb.Dataset(
                X_val_model,
                label=y_val[:, h],
                weight=val_weight,
                reference=dtrain,
                categorical_feature=CATE_VARS,
                free_raw_data=True
            )

            model = lgb.train(
                params,
                dtrain,
                num_boost_round=OPTUNA_MAX_ROUNDS,
                valid_sets=[dval],
                valid_names=["val"],
                callbacks=[
                    lgb.early_stopping(OPTUNA_EARLY_STOPPING, verbose=False),
                    lgb.log_evaluation(period=0),
                ]
            )

            best_iter = int(model.best_iteration or OPTUNA_MAX_ROUNDS)
            best_iters_tmp.append(best_iter)

            val_pred_tmp[:, j] = model.predict(
                X_val_model,
                num_iteration=best_iter
            ).astype(np.float32)

            del dtrain, dval, model
            gc.collect()

        y_val_tmp = y_val[:, TUNE_HORIZONS]

        err = (y_val_tmp - val_pred_tmp) ** 2
        score = float(
            np.sqrt(
                (err.sum(axis=1) * val_weight).sum()
                / val_weight.sum()
                / len(TUNE_HORIZONS)
            )
        )

        trial.set_user_attr("best_iterations", best_iters_tmp)

        return score

    study = optuna.create_study(direction="minimize")
    study.optimize(
        objective,
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    print("\n✅ Optuna best score:", study.best_value)
    print("✅ Optuna best params:")
    print(study.best_params)

    OPTUNA_BEST_PARAMS = BASE_PARAMS.copy()
    OPTUNA_BEST_PARAMS.update(study.best_params)

    # Fix lại các tham số bắt buộc
    OPTUNA_BEST_PARAMS.update({
        "objective": "regression",
        "metric": "l2",
        "verbosity": -1,
        "num_threads": -1,
        "force_col_wise": True,

        "seed": SEED,
        "feature_fraction_seed": SEED,
        "bagging_seed": SEED,
        "data_random_seed": SEED,
    })

    out_path = os.path.join(WORKING_DIR, "optuna_best_params.json")
    with open(out_path, "w") as f:
        json.dump(OPTUNA_BEST_PARAMS, f, indent=2)

    print("\n💾 Đã lưu Optuna params mới tại:")
    print(out_path)

    print("\n✅ OPTUNA_BEST_PARAMS sau tuning:")
    for k, v in OPTUNA_BEST_PARAMS.items():
        print(f"{k}: {v}")

    del X_train_opt, y_train_opt, train_weight_opt
    gc.collect()

✅ Tìm thấy file Optuna params đã tune sẵn.
📌 Đường dẫn: /kaggle/input/datasets/hailehong/optuna/optuna_best_params.json

✅ OPTUNA_BEST_PARAMS được load từ file:
objective: regression
metric: l2
learning_rate: 0.015617477447337045
num_leaves: 96
max_depth: 13
min_data_in_leaf: 150
feature_fraction: 0.6344755945104896
bagging_fraction: 0.8527008548252045
bagging_freq: 2
lambda_l1: 0.7697964384041276
lambda_l2: 0.51820780127672
min_gain_to_split: 0.20142981269439797
max_bin: 255
verbosity: -1
num_threads: -1
force_col_wise: True
seed: 2026
feature_fraction_seed: 2026
bagging_seed: 2026
data_random_seed: 2026

💾 Đã lưu bản copy tại:
/kaggle/working/optuna_best_params_loaded.json


In [6]:
# ============================================================
# CELL 6: TRAIN 16 LIGHTGBM MODELS BẰNG OPTUNA_BEST_PARAMS
# ============================================================

print("🚀 Bắt đầu train 16 LightGBM models...")

if "OPTUNA_BEST_PARAMS" not in globals():
    raise RuntimeError(
        "Chưa có OPTUNA_BEST_PARAMS. "
        "Hãy chạy Cell 5 Optuna trước khi chạy Cell 6."
    )

FINAL_PARAMS = OPTUNA_BEST_PARAMS.copy()

print("✅ Dùng tham số đã tinh chỉnh bằng Optuna.")
print("\nFINAL_PARAMS:")
for k, v in FINAL_PARAMS.items():
    print(f"{k}: {v}")

# False: train có early stopping rồi predict test luôn
# True : sau khi có best_iteration, refit trên train + val rồi predict test
DO_FINAL_REFIT = False

val_pred = np.zeros((X_val_model.shape[0], 16), dtype=np.float32)
test_pred = np.zeros((X_test_model.shape[0], 16), dtype=np.float32)

best_iterations = []
models = []

for h in range(16):
    print("=" * 80)
    print(f" Horizon {h + 1}/16")
    print("=" * 80)

    dtrain = lgb.Dataset(
        X_train_model,
        label=y_train[:, h],
        weight=train_weight,
        categorical_feature=CATE_VARS,
        free_raw_data=True
    )

    dval = lgb.Dataset(
        X_val_model,
        label=y_val[:, h],
        weight=val_weight,
        reference=dtrain,
        categorical_feature=CATE_VARS,
        free_raw_data=True
    )

    model = lgb.train(
        FINAL_PARAMS,
        dtrain,
        num_boost_round=MAX_ROUNDS,
        valid_sets=[dtrain, dval],
        valid_names=["train", "val"],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=True),
            lgb.log_evaluation(period=100),
        ]
    )

    best_iter = int(model.best_iteration or MAX_ROUNDS)
    best_iterations.append(best_iter)

    val_pred[:, h] = model.predict(
        X_val_model,
        num_iteration=best_iter
    ).astype(np.float32)

    if DO_FINAL_REFIT:
        print(f"🔁 Refit horizon {h + 1}/16 trên train + val, rounds = {best_iter}")

        X_all = pd.concat(
            [X_train_model, X_val_model],
            axis=0,
            ignore_index=True
        )

        y_all_h = np.concatenate(
            [y_train[:, h], y_val[:, h]]
        ).astype(np.float32)

        w_all = np.concatenate(
            [train_weight, val_weight]
        ).astype(np.float32)

        dall = lgb.Dataset(
            X_all,
            label=y_all_h,
            weight=w_all,
            categorical_feature=CATE_VARS,
            free_raw_data=True
        )

        final_model = lgb.train(
            FINAL_PARAMS,
            dall,
            num_boost_round=best_iter,
            callbacks=[lgb.log_evaluation(period=0)]
        )

        test_pred[:, h] = final_model.predict(
            X_test_model,
            num_iteration=best_iter
        ).astype(np.float32)

        del X_all, y_all_h, w_all, dall, final_model
        gc.collect()

    else:
        test_pred[:, h] = model.predict(
            X_test_model,
            num_iteration=best_iter
        ).astype(np.float32)

    gain = model.feature_importance("gain")
    top_features = sorted(
        zip(X_train_model.columns, gain),
        key=lambda x: x[1],
        reverse=True
    )[:25]

    print("\nTop 25 features:")
    for name, score in top_features:
        print(f"{name}: {score:.2f}")

    models.append(model)

    del dtrain, dval, model
    gc.collect()

val_score = nwrmsle_log(y_val, val_pred, val_weight)

print("=" * 80)
print(f"✅ Validation NWRMSLE/log weighted: {val_score:.6f}")
print("✅ Best iterations:", best_iterations)
print("=" * 80)

np.save(os.path.join(WORKING_DIR, "lgbm_val_pred_log.npy"), val_pred)
np.save(os.path.join(WORKING_DIR, "lgbm_test_pred_log.npy"), test_pred)
np.save(
    os.path.join(WORKING_DIR, "lgbm_best_iterations.npy"),
    np.array(best_iterations, dtype=np.int32)
)

with open(os.path.join(WORKING_DIR, "lgbm_final_params.json"), "w") as f:
    json.dump(FINAL_PARAMS, f, indent=2)

print("💾 Đã lưu:")
print("- lgbm_val_pred_log.npy")
print("- lgbm_test_pred_log.npy")
print("- lgbm_best_iterations.npy")
print("- lgbm_final_params.json")

🚀 Bắt đầu train 16 LightGBM models...
✅ Dùng tham số đã tinh chỉnh bằng Optuna.

FINAL_PARAMS:
objective: regression
metric: l2
learning_rate: 0.015617477447337045
num_leaves: 96
max_depth: 13
min_data_in_leaf: 150
feature_fraction: 0.6344755945104896
bagging_fraction: 0.8527008548252045
bagging_freq: 2
lambda_l1: 0.7697964384041276
lambda_l2: 0.51820780127672
min_gain_to_split: 0.20142981269439797
max_bin: 255
verbosity: -1
num_threads: -1
force_col_wise: True
seed: 2026
feature_fraction_seed: 2026
bagging_seed: 2026
data_random_seed: 2026
 Horizon 1/16
Training until validation scores don't improve for 150 rounds
[100]	train's l2: 0.335871	val's l2: 0.33019
[200]	train's l2: 0.28977	val's l2: 0.288797
[300]	train's l2: 0.281776	val's l2: 0.283414
[400]	train's l2: 0.278053	val's l2: 0.281891
[500]	train's l2: 0.275373	val's l2: 0.281136
[600]	train's l2: 0.273085	val's l2: 0.280667
[700]	train's l2: 0.271045	val's l2: 0.28034
[800]	train's l2: 0.269202	val's l2: 0.28009
[900]	train's

In [7]:
# ============================================================
# CELL 7: TẠO SUBMISSION ĐÚNG 3,370,464 DÒNG
# ============================================================

import os
import gc
import subprocess
import numpy as np
import pandas as pd

print("🚀 Đang tạo submission đúng theo full test.csv...")

WORKING_DIR = "/kaggle/working"


# ============================================================
# 1. LOAD TEST PRED
# ============================================================

if "test_pred" not in globals():
    test_pred = np.load(os.path.join(WORKING_DIR, "lgbm_test_pred_log.npy"))

print("test_pred:", test_pred.shape)

if "X_test" not in globals() and "X_test_model" not in globals():
    raise RuntimeError("Không tìm thấy X_test hoặc X_test_model trong RAM.")

# Ưu tiên lấy index từ test_id_matrix nếu có MultiIndex đúng
if "test_id_matrix" in globals() and isinstance(test_id_matrix.index, pd.MultiIndex):
    pred_index = test_id_matrix.index
else:
    if isinstance(X_test.index, pd.MultiIndex):
        pred_index = X_test.index
    else:
        raise RuntimeError(
            "Không tìm thấy MultiIndex store_nbr-item_nbr cho X_test. "
            "Cần X_test hoặc test_id_matrix có index là MultiIndex."
        )

if test_pred.shape[0] != len(pred_index):
    raise ValueError(
        f"Số dòng test_pred không khớp pred_index: "
        f"{test_pred.shape[0]} vs {len(pred_index)}"
    )


# ============================================================
# 2. TẠO PREDICTION LONG TABLE TỪ TEST_PRED
# ============================================================

# test_pred đang là log1p prediction
pred_sales = np.clip(np.expm1(test_pred), 0, 1000).astype(np.float32)

# Xác định 16 ngày test
test_dates = pd.date_range("2017-08-16", periods=16)

pred_df = pd.DataFrame(
    pred_sales,
    index=pred_index,
    columns=test_dates
)

pred_long = pred_df.stack(dropna=False).rename("unit_sales").reset_index()

# Chuẩn hoá tên cột
pred_long.columns = ["store_nbr", "item_nbr", "date", "unit_sales"]

pred_long["date"] = pd.to_datetime(pred_long["date"])
pred_long["store_nbr"] = pred_long["store_nbr"].astype(np.int16)
pred_long["item_nbr"] = pred_long["item_nbr"].astype(np.int32)

print("pred_long:", pred_long.shape)


# ============================================================
# 3. LOAD FULL TEST.CSV GỐC ĐỂ LẤY ĐỦ TOÀN BỘ ID
# ============================================================

def find_file(filename, root="/kaggle/input"):
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return os.path.join(dirpath, filename)
    return None

test_csv_path = find_file("test.csv", "/kaggle/input")

# Nếu input là test.csv.7z thì giải nén
if test_csv_path is None:
    test_7z_path = find_file("test.csv.7z", "/kaggle/input")
    if test_7z_path is None:
        raise FileNotFoundError(
            "Không tìm thấy test.csv hoặc test.csv.7z trong /kaggle/input. "
            "Hãy add original Favorita dataset vào notebook."
        )

    print("📦 Tìm thấy test.csv.7z:", test_7z_path)
    print("⏳ Đang giải nén test.csv.7z vào /kaggle/working ...")

    subprocess.run(
        ["7z", "x", test_7z_path, f"-o{WORKING_DIR}", "-y"],
        check=True
    )

    test_csv_path = os.path.join(WORKING_DIR, "test.csv")

print("✅ Full test path:", test_csv_path)

df_test_full = pd.read_csv(
    test_csv_path,
    usecols=["id", "date", "store_nbr", "item_nbr"],
    dtype={
        "id": "int64",
        "store_nbr": "int16",
        "item_nbr": "int32"
    },
    parse_dates=["date"]
)

print("df_test_full:", df_test_full.shape)

expected_rows = 3370464
if len(df_test_full) != expected_rows:
    print(
        f"⚠️ Cảnh báo: full test có {len(df_test_full)} dòng, "
        f"khác expected {expected_rows}. Vẫn tiếp tục theo test.csv."
    )


# ============================================================
# 4. MERGE PREDICTION VÀO FULL TEST, FILL MISSING = 0
# ============================================================

submission = df_test_full.merge(
    pred_long,
    on=["store_nbr", "item_nbr", "date"],
    how="left"
)

missing_pred = submission["unit_sales"].isna().sum()
print("Số dòng không có prediction, sẽ fill 0:", missing_pred)

submission["unit_sales"] = submission["unit_sales"].fillna(0).astype(np.float32)

submission = submission[["id", "unit_sales"]].sort_values("id")

OUTPUT_FILE = os.path.join(WORKING_DIR, "submission_lightgbm_txla.csv")
submission.to_csv(OUTPUT_FILE, index=False, float_format="%.4f")


# ============================================================
# 5. SANITY CHECK
# ============================================================

print("\n✅ Đã tạo submission:", OUTPUT_FILE)
print("Shape:", submission.shape)
print(submission.head())
print(submission.tail())

if len(submission) != expected_rows:
    raise ValueError(
        f"Submission sai số dòng: {len(submission)}. "
        f"Kaggle yêu cầu {expected_rows}."
    )

if submission["id"].duplicated().sum() > 0:
    raise ValueError("Submission có id bị trùng.")

if submission["unit_sales"].isna().sum() > 0:
    raise ValueError("Submission có unit_sales bị NaN.")

print("\n✅ Submission hợp lệ để nộp Kaggle.")
print("Rows:", len(submission))
print("Duplicate id:", submission["id"].duplicated().sum())
print("NaN unit_sales:", submission["unit_sales"].isna().sum())

gc.collect()

🚀 Đang tạo submission đúng theo full test.csv...
test_pred: (167515, 16)
pred_long: (2680240, 4)
✅ Full test path: /kaggle/input/notebooks/luminhanh/ba-model-prep-data/test.csv
df_test_full: (3370464, 4)
Số dòng không có prediction, sẽ fill 0: 795040

✅ Đã tạo submission: /kaggle/working/submission_lightgbm_txla.csv
Shape: (3370464, 2)
          id  unit_sales
0  125497040    0.204808
1  125497041    0.225120
2  125497042    0.000000
3  125497043    1.119949
4  125497044    1.830095
                id  unit_sales
3370459  128867499         0.0
3370460  128867500         0.0
3370461  128867501         0.0
3370462  128867502         0.0
3370463  128867503         0.0

✅ Submission hợp lệ để nộp Kaggle.
Rows: 3370464
Duplicate id: 0
NaN unit_sales: 0


0